# Testing a pre-processing idea

One harness, one protocol. You change **only** the pre-processing function. The split, the
targets, the de-duplication, the threshold and the models are identical for all four of us,
which is what makes your number comparable to mine.

Full notes: `docs/HOW_TO_RUN_EXPERIMENTS.md`

## What is frozen

| | |
|---|---|
| split | `E.setup(seed=42, val_size=0.2)`, grouped on raw CRM configuration |
| targets | consensus targets from raw `train.csv` -- scoring never sees your pre-processing |
| de-duplication | on the **raw** configuration, *before* your function runs |
| threshold | 0.5 |
| additive rules | applied to the **raw** validation CRM matrix |

De-duplication happens before your function so every idea trains on the same 53,947 rows and
only the features vary. Otherwise a transform that merges configurations would silently change
the training set size, and "better features" could not be told apart from "different data".

## Baselines to beat (pre-processing = identity, seed 42)

| model | no rules | with rules |
|---|---|---|
| `ovr-l1-lr` | - | 86.4367 % |
| `cooc-chain-lgbm` | 87.5141 % | 87.7092 % |
| `br-lgbm` | 87.6543 % | **87.8411 %** |

A difference under about 0.05 points is probably noise. The Sprint 2 champion decision turned
on 0.022 points and that was judged not significant (p = 0.096).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# the harness lives at the repo root, one level above notebooks/
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from testing_pipeline import LOG_FILE, run_experiment

print('repo root:', ROOT)

## Step 1 -- write your idea

The function receives the training features, the training labels and the validation features,
and returns all three, transformed.

- `X_train` is `(53947, 745)` uint8, one row per unique CRM configuration, all values 0/1
- `Y_train` is `(53947, 731)` uint8
- `X_val` is `(36385, 745)` uint8

**Two rules you must not break.**

1. Whatever you do to a training column, do to the validation column. The harness checks the
   column *counts* match but cannot check you dropped the *same* columns -- that one is on you.
2. Never add or drop validation rows. Scoring is against a fixed 36,385 rows; filtering them
   means scoring on an easier set. The harness rejects this.

You *may* drop or add training rows -- outlier removal and augmentation are legitimate ideas.
Always `np.copy` first so you do not mutate the arrays in place.

In [ ]:
def my_idea(X_train, Y_train, X_val):
    """One sentence saying what you think will help and why.

    EDIT THIS. As written it is the identity transform, which reproduces the baselines above.
    """
    X_tr = np.copy(X_train)
    Y_tr = np.copy(Y_train)
    X_va = np.copy(X_val)

    # ---- your transformation goes here ----
    # example: mask a column in BOTH train and validation
    #     X_tr[:, 10] = 0
    #     X_va[:, 10] = 0
    # example: drop training rows only
    #     keep = X_tr.sum(axis=1) > 2
    #     X_tr, Y_tr = X_tr[keep], Y_tr[keep]

    return X_tr, Y_tr, X_va


AUTHOR = 'YourName'
IDEA_NAME = 'Short_Name_For_The_Idea'

## Step 2 -- run it

Set `AUTHOR` and `IDEA_NAME` above before running, or your result lands in the shared log as
`YourName / Short_Name_For_The_Idea`.

Pass one model while iterating; all three cost about 10 minutes.

| | cost |
|---|---|
| `br-lgbm` | ~210 s |
| `ovr-l1-lr` | ~208 s |
| `cooc-chain-lgbm` | ~190 s |
| `full_metrics=True` | +5 s per rules variant |
| `compute_auc=True` | +10 s per model |
| re-running an unchanged idea | ~15 s (cached) |

**The very first run of the session takes about 4 extra minutes** while `data/cache/arrays.npz`
is built from the raw CSVs. After that it loads in seconds.

Probabilities are cached by a content hash of the arrays, so re-running an unchanged idea loads
from disk, and editing your function invalidates the cache automatically -- you can never get a
stale number from a previous idea.

In [ ]:
run_experiment(
    author=AUTHOR,
    idea_name=IDEA_NAME,
    preprocess_func=my_idea,
    models='br-lgbm',   # None runs all three
    rules='both',       # True, False, or 'both'
)

## Step 3 -- read the result

**EMR is the only metric that counts.** It is the competition metric: a row scores only if all
731 BIL bits are right. Everything else is a diagnostic.

Do **not** optimise F1 or Hamming loss -- they actively disagree with EMR on this problem.
Applying the 42 additive rules raises EMR by 0.19 points while making F1 micro, precision and
Hamming loss all worse, because it adds 1,358 false positives to fix 73 false negatives on rows
that were one bit from perfect. Optimise F1 and you will throw away a real gain.

`EMR_vs_raw` is EMR against the original training labels rather than the consensus ones. Quote
it when comparing to the test set, since `solution.csv` holds raw labels.

In [ ]:
BASELINE = {('br-lgbm', False): 0.8765425, ('br-lgbm', True): 0.8784114,
            ('cooc-chain-lgbm', False): 0.8751409, ('cooc-chain-lgbm', True): 0.8770922,
            ('ovr-l1-lr', True): 0.8643672}

log = pd.read_csv(LOG_FILE)
log['Baseline_EMR'] = [BASELINE.get((m, r), np.nan)
                       for m, r in zip(log['Model'], log['Rules_Applied'])]
log['dEMR_pts'] = (log['EMR'] - log['Baseline_EMR']) * 100

cols = ['Author', 'Idea_Name', 'Model', 'Rules_Applied', 'EMR', 'dEMR_pts',
        'EMR_vs_raw', 'F1_micro', 'Train_Rows']
log[cols].sort_values('EMR', ascending=False)

A positive `dEMR_pts` under about 0.05 is most likely noise rather than a real gain. Before
claiming an improvement, ask for a paired significance test -- the cached probabilities make it
a seconds-long check rather than a refit.